In [1]:
# Cell 1 — Install (restart kernel after this cell)
import subprocess
subprocess.run(['pip', 'uninstall', 'unsloth', 'unsloth_zoo', 'torchao', 'trl', 'transformers', 'bitsandbytes', '-y'])
subprocess.run(['pip', 'install', '-q',
    'transformers==4.46.3',
    'trl==0.12.2',
    'peft==0.13.2',
    'accelerate==1.1.1',
    'datasets',
    'tqdm',
    'faiss-cpu',
    'sentence-transformers',
])
print('Done. Restart kernel now, then run all cells below.')

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 87.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 76.9 MB/s eta 0:00:00
Done. Restart kernel now, then run all cells below.


In [2]:
# Cell 2 — Imports
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import random
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from datasets import load_dataset, Dataset
from sentence_transformers import SentenceTransformer
import faiss
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig

random.seed(42)
torch.manual_seed(42)
print('Imports OK')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

2026-05-23 00:27:33.057739: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779496053.295215      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779496053.355771      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779496053.895971      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779496053.896014      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779496053.896017      23 computation_placer.cc:177] computation placer alr

Imports OK
PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
Free: 15.5 GB


In [3]:
# Cell 3 — Load all 4 English datasets
say = load_dataset('jjzha/sayfullina')
skl = load_dataset('jjzha/skillspan')
grn = load_dataset('jjzha/green')
gnh = load_dataset('jjzha/gnehm')

for name, ds in [('sayfullina', say), ('skillspan', skl), ('green', grn), ('gnehm', gnh)]:
    print(f'{name:12} train={len(ds["train"]):5}  test={len(ds["test"]):5}')

README.md: 0.00B [00:00, ?B/s]

train.json: 0.00B [00:00, ?B/s]

dev.json: 0.00B [00:00, ?B/s]

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3705 [00:00<?, ? examples/s]

...

test.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/19889 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2332 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2557 [00:00<?, ? examples/s]

sayfullina   train= 3705  test= 1851
skillspan    train= 4800  test= 3569
green        train= 8669  test=  335
gnehm        train=19889  test= 2557


In [4]:
# Cell 4 — Tag normalisation + BIO helpers
def normalise_tag(tag):
    if tag in ('B', 'B-SOFT', 'B-ICT', 'B-SKILL',
               'B-PENSEE', 'B-RESULTATS', 'B-RELATIONNEL', 'B-PERSONNEL'):
        return 'B-SKILL'
    if tag in ('I', 'I-SOFT', 'I-ICT', 'I-SKILL',
               'I-PENSEE', 'I-RESULTATS', 'I-RELATIONNEL', 'I-PERSONNEL'):
        return 'I-SKILL'
    return 'O'

def get_tokens_and_tags(example):
    """Merges tags_skill + tags_knowledge (where present) into B-SKILL/I-SKILL/O."""
    tokens = example['tokens']
    merged = ['O'] * len(tokens)
    for col in ('tags_skill', 'tags_knowledge'):
        if col not in example:
            continue
        for i, tag in enumerate(example[col]):
            norm = normalise_tag(str(tag))
            if norm != 'O' and merged[i] == 'O':
                merged[i] = norm
    return tokens, merged

def to_bracket_format(tokens, tags):
    result = []
    i = 0
    while i < len(tokens):
        if tags[i] == 'B-SKILL':
            span = [tokens[i]]
            j = i + 1
            while j < len(tokens) and tags[j] == 'I-SKILL':
                span.append(tokens[j])
                j += 1
            result.append(f"@@{' '.join(span)}##")
            i = j
        else:
            result.append(tokens[i])
            i += 1
    return ' '.join(result)

def spans_from_bio(tokens, tags):
    spans, current = [], []
    for token, tag in zip(tokens, tags):
        if tag == 'B-SKILL':
            if current:
                spans.append(' '.join(current))
            current = [token]
        elif tag == 'I-SKILL' and current:
            current.append(token)
        else:
            if current:
                spans.append(' '.join(current))
            current = []
    if current:
        spans.append(' '.join(current))
    return spans

print('Helpers defined')

Helpers defined


In [5]:
# Cell 5 — Cap training examples per dataset + sanity check
MAX_TRAIN_PER_DATASET = 15000  # lower to 200 for ~1.5h, raise to 1000 for ~7h

def cap_dataset(ds, n):
    return ds['train'].shuffle(seed=42).select(range(min(n, len(ds['train']))))

say_train = cap_dataset(say, MAX_TRAIN_PER_DATASET)
skl_train = cap_dataset(skl, MAX_TRAIN_PER_DATASET)
grn_train = cap_dataset(grn, MAX_TRAIN_PER_DATASET)
gnh_train = cap_dataset(gnh, MAX_TRAIN_PER_DATASET)

for name, split in [('sayfullina', say_train), ('skillspan', skl_train), ('green', grn_train), ('gnehm', gnh_train)]:
    skill_count = sum(1 for ex in split if any(t != 'O' for t in get_tokens_and_tags(ex)[1]))
    print(f'{name:12} train={len(split)}  with_skills={skill_count}')

sayfullina   train=3705  with_skills=3701
skillspan    train=4800  with_skills=1584
green        train=8669  with_skills=5136
gnehm        train=15000  with_skills=2587


In [6]:
# Cell 6 — System prompt
SYSTEM_ICL = (
    'You are a skill-span extractor specialising in job advertisement sentences. '
    'Your task is to identify ALL skill spans — both hard skills (tools, technologies, '
    'programming languages, domain knowledge) and soft skills (interpersonal, behavioural, '
    'personal competencies such as communication, leadership, teamwork). '
    'Sentences may be in any language and may be grammatically incomplete. '
    'Extract spans exactly as they appear, token-by-token. '
    'Wrap each skill span with @@...## brackets. '
    'Copy the sentence exactly with no other changes. '
    'If there are no skills, return the sentence unchanged. '
    'Output only the annotated sentence — no explanation, no commentary.'
)

def format_prompt(user_text, target=None):
    prompt = (
        f'<|im_start|>system\n{SYSTEM_ICL}<|im_end|>\n'
        f'<|im_start|>user\n{user_text}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    if target is not None:
        prompt += f'{target}<|im_end|>'
    return prompt

print('Prompt defined')

Prompt defined


In [7]:
# Cell 7 — Build RAG-1 index (all 4 datasets combined, capped)
embedder = SentenceTransformer('all-MiniLM-L6-v2')

ALL_DATASETS = [
    ('sayfullina', say_train),
    ('skillspan',  skl_train),
    ('green',      grn_train),
    ('gnehm',      gnh_train),
]

all_train_with_skills = []
for name, split in ALL_DATASETS:
    count = 0
    for ex in split:
        tokens, tags = get_tokens_and_tags(ex)
        if any(t != 'O' for t in tags):
            all_train_with_skills.append({'tokens': tokens, 'tags': tags, 'source': name})
            count += 1
    print(f'{name}: {count} examples with skills')

print(f'\nTotal RAG-1 pool: {len(all_train_with_skills)}')

rag1_sentences = [' '.join(ex['tokens']) for ex in all_train_with_skills]
print('Embedding RAG-1 sentences...')
rag1_emb = embedder.encode(rag1_sentences, show_progress_bar=True, batch_size=64).astype('float32')
faiss.normalize_L2(rag1_emb)
rag1_index = faiss.IndexFlatIP(rag1_emb.shape[1])
rag1_index.add(rag1_emb)
print(f'RAG-1 index: {rag1_index.ntotal} entries')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

sayfullina: 3701 examples with skills
skillspan: 1584 examples with skills
green: 5136 examples with skills
gnehm: 2587 examples with skills

Total RAG-1 pool: 13008
Embedding RAG-1 sentences...


Batches:   0%|          | 0/204 [00:00<?, ?it/s]

RAG-1 index: 13008 entries


In [8]:
# Cell 8 — Build RAG-2 ESCO index
ESCO_PATH = None
for root, _, files in os.walk('/kaggle/input'):
    for f in files:
        if f == 'skills_en.csv':
            ESCO_PATH = os.path.join(root, f)
            break
    if ESCO_PATH:
        break

if ESCO_PATH is None:
    raise FileNotFoundError('skills_en.csv not found. Add the ESCO dataset via the Data panel.')
print(f'ESCO path: {ESCO_PATH}')

esco = pd.read_csv(ESCO_PATH)

def build_esco_text(row):
    parts = [str(row.get('preferredLabel', ''))]
    alt = str(row.get('altLabels', ''))
    if alt and alt != 'nan':
        parts.append(alt.replace('\n', ', '))
    desc = str(row.get('description', ''))
    if desc and desc != 'nan':
        parts.append(desc[:200])
    return ' | '.join(parts)

esco_texts  = [build_esco_text(row) for _, row in esco.iterrows()]
esco_labels = esco['preferredLabel'].tolist()

print('Embedding ESCO skills...')
rag2_emb = embedder.encode(esco_texts, show_progress_bar=True, batch_size=64).astype('float32')
faiss.normalize_L2(rag2_emb)
rag2_index = faiss.IndexFlatIP(rag2_emb.shape[1])
rag2_index.add(rag2_emb)
print(f'RAG-2 index: {rag2_index.ntotal} entries')

ESCO path: /kaggle/input/datasets/acme105/skills-en-esco/skills_en.csv
Embedding ESCO skills...


Batches:   0%|          | 0/219 [00:00<?, ?it/s]

RAG-2 index: 13960 entries


In [9]:
# Cell 9 — RAG retrieval + prompt builder
def retrieve_rag1(query_tokens, k=3):
    q_emb = embedder.encode([' '.join(query_tokens)]).astype('float32')
    faiss.normalize_L2(q_emb)
    _, indices = rag1_index.search(q_emb, k)
    return [(all_train_with_skills[i]['tokens'], all_train_with_skills[i]['tags'])
            for i in indices[0]]

def retrieve_rag2(candidate_spans, k=3):
    if not candidate_spans:
        return ''
    definitions = []
    for span in candidate_spans[:5]:
        q_emb = embedder.encode([span]).astype('float32')
        faiss.normalize_L2(q_emb)
        _, indices = rag2_index.search(q_emb, k)
        for idx in indices[0]:
            desc = str(esco.iloc[idx].get('description', ''))
            if desc and desc != 'nan':
                definitions.append(f'- {esco_labels[idx]}: {desc[:120]}')
    seen, unique = set(), []
    for d in definitions:
        if d not in seen:
            seen.add(d)
            unique.append(d)
    return '\n'.join(unique[:8])

def build_user_text(query_tokens, rag1_examples, esco_context):
    lines = []
    if rag1_examples:
        lines.append('Examples of skill annotation:')
        for ex_tokens, ex_tags in rag1_examples:
            lines.append(f'  Input:  {" ".join(ex_tokens)}')
            lines.append(f'  Output: {to_bracket_format(ex_tokens, ex_tags)}')
        lines.append('')
    if esco_context:
        lines.append('Relevant ESCO skill definitions:')
        lines.append(esco_context)
        lines.append('')
    lines.append(' '.join(query_tokens))
    return '\n'.join(lines)

print('RAG functions defined')

RAG functions defined


In [10]:
# Cell 10 — Build RAG-augmented SFT training set (all 4 datasets, capped)
def build_training_set():
    positives, negatives = [], []

    for name, split in ALL_DATASETS:
        for ex in tqdm(split, desc=f'Building {name}'):
            tokens, tags = get_tokens_and_tags(ex)
            target = to_bracket_format(tokens, tags)

            rag1_examples = retrieve_rag1(tokens, k=4)
            self_sentence = ' '.join(tokens)
            rag1_examples = [
                (t, tg) for t, tg in rag1_examples
                if ' '.join(t) != self_sentence
            ][:3]

            gold_spans   = spans_from_bio(tokens, tags)
            esco_context = retrieve_rag2(gold_spans if gold_spans else [' '.join(tokens)], k=2)

            user_text = build_user_text(tokens, rag1_examples, esco_context)
            entry = {'text': format_prompt(user_text, target)}

            if '@@' in target:
                positives.append(entry)
            else:
                negatives.append(entry)

    max_neg   = int(len(positives) * 0.3)
    negatives = random.sample(negatives, min(max_neg, len(negatives)))
    combined  = positives + negatives
    random.shuffle(combined)
    print(f'Positives: {len(positives)}, Hard negatives: {len(negatives)}, Total: {len(combined)}')
    return combined

train_data = build_training_set()
hf_train   = Dataset.from_list(train_data)
print('\nSample prompt (truncated):')
print(train_data[0]['text'][:600])

Building gnehm: 100%|██████████| 15000/15000 [04:28<00:00, 55.84it/s]


Positives: 12579, Hard negatives: 3773, Total: 16352

Sample prompt (truncated):
<|im_start|>system
You are a skill-span extractor specialising in job advertisement sentences. Your task is to identify ALL skill spans — both hard skills (tools, technologies, programming languages, domain knowledge) and soft skills (interpersonal, behavioural, personal competencies such as communication, leadership, teamwork). Sentences may be in any language and may be grammatically incomplete. Extract spans exactly as they appear, token-by-token. Wrap each skill span with @@...## brackets. Copy the sentence exactly with no other changes. If there are no skills, return the sentence unchange


In [11]:
# Cell 11 — Load model + LoRA
import gc
gc.collect()
torch.cuda.empty_cache()
print(f'Free GPU: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map={'': 'cuda:0'},
    trust_remote_code=True,
)
model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.enable_input_require_grads()
print(f'GPU after LoRA: {torch.cuda.memory_allocated()/1e9:.1f} GB')

Free GPU: 15.4 GB


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
GPU after LoRA: 6.4 GB


In [12]:
# Cell 12 — Train
sft_config = SFTConfig(
    output_dir                    = '/kaggle/working/checkpoints',
    per_device_train_batch_size   = 2,
    gradient_accumulation_steps   = 8,
    num_train_epochs              = 2,
    learning_rate                 = 2e-4,
    warmup_ratio                  = 0.1,
    lr_scheduler_type             = 'cosine',
    fp16                          = True,
    gradient_checkpointing        = True,
    gradient_checkpointing_kwargs = {'use_reentrant': False},
    logging_steps                 = 20,
    save_strategy                 = 'epoch',
    report_to                     = 'none',
    seed                          = 42,
    optim                         = 'adamw_torch',
    ddp_find_unused_parameters    = False,
    dataloader_pin_memory         = False,
    dataset_text_field            = 'text',
    max_seq_length                = 512,
)

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = hf_train,
    args          = sft_config,
)

trainer.train()
print('Training complete.')

Map:   0%|          | 0/16352 [00:00<?, ? examples/s]

Step,Training Loss
20,3.245500
40,2.019500
60,1.343400
80,1.179600
100,1.131800
120,1.079700
140,1.062000
160,1.022300
180,1.028100
200,0.985500


Training complete.


In [13]:
# Cell 13 — Save adapter
SAVE_PATH = '/kaggle/working/lora_adapter_p7'
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print('Saved:', os.listdir(SAVE_PATH))

Saved: ['added_tokens.json', 'adapter_config.json', 'vocab.json', 'merges.txt', 'adapter_model.safetensors', 'special_tokens_map.json', 'tokenizer_config.json', 'tokenizer.json', 'README.md']
